# 02 — TXS 0506+056: the Fermi-LAT counterpart to IceCube-170922A

On 2017-09-22 IceCube reported a 290 TeV neutrino — **IC-170922A** — whose 90 % containment region landed on top of the BL Lac **TXS 0506+056** ([4FGL J0509.4+0542](https://fermi.gsfc.nasa.gov/cgi-bin/ssc/LAT/14yr_catalog/source.cgi?source=J0509.4+0542), z = 0.3365). The follow-up Fermi-LAT analysis [IceCube et al. 2018, *Science* 361, eaat1378](https://www.science.org/doi/10.1126/science.aat1378) showed the source in a **6-month gamma-ray flare** at the time of the alert — the first compelling extragalactic neutrino source identification.

This notebook redoes the LAT side of that analysis end-to-end:

| | |
|---|---|
| Center | (RA, Dec) = (77.3582°, +5.69315°) |
| ROI radius (data) | 15° |
| Time window | MET 494 380 804 → 557 452 805  (2016-09-01 → 2018-09-01, alert at the centre) |
| Energy | 100 MeV – 300 GeV |
| Catalog | 4FGL (DR4) |


## Why TXS 0506+056 is *harder* than PG 1553+113

### 1. Galactic latitude

| Source | Galactic (l, b) | csc \|b\| (rough diffuse-brightness proxy) |
|---|---|---|
| PG 1553+113 | (21.9°,  +44.0°) | 1.44 |
| **TXS 0506+056** | (195.4°, **−19.6°**) | **2.97** |

TXS sits ~24° closer to the Galactic plane than PG 1553. The Galactic diffuse emission (`gll_iem_v07.fits`) scales roughly as csc \|b\| at moderate latitudes, so the diffuse background under TXS is **~2× brighter** than under PG 1553 — and, worse, it has a **strong gradient across the 10°×10° fit ROI** because the low-b edge of the ROI sits at b ≈ −5°, only 5° from the plane. Any mis-modelling of that gradient masquerades as flux from the target. This could be worse (we could be *within* the plane), but it is something to keep in mind.


### 2. Source crowding & PSF-mixing at low energy

TXS lies toward the Galactic anti-centre (l ≈ 195°), so the inner-Galaxy diffuse component is suppressed, but it's still in a busier 4FGL neighborhood than PG 1553. The LAT PSF at 100 MeV is ~3–5°, larger than the typical separation of catalog sources in this field. Below ~300 MeV, every photon's source assignment is statistical, and a poorly-fit neighbour will leak counts into the target.

### 3. Variability

TXS is a **flaring blazar**. The 2-year window we are fitting straddles a months-long flare around the IceCube alert. A single-state spectral fit averages a quiescent ~10⁻⁸ ph cm⁻² s⁻¹ baseline with a flare ~5–10× brighter, so:

- the LogParabola curvature parameter tries to compensate for two states with one curve and ends up poorly determined;
- the average spectrum is **not** the spectrum *during* the flare. To see the flare we have to do a **lightcurve** — which is the last section.

### 4. EBL

At z = 0.3365 the EBL optical depth τ_γγ reaches ≈1 around 150–200 GeV, so the topmost bin (300 GeV) is already attenuated. We fit the *observed* spectrum and don't correct for EBL — just remember that the highest-energy bin is biased low. (For a proper intrinsic-SED measurement you'd attach an EBL absorption model to the source spectrum.)


## Setup

Same conventions as notebook 01: working directory under `data/txs0506/`, `FERMI_DIFFUSE_DIR` pointed at the diffuse files inside the conda env.


In [ ]:
import os
from pathlib import Path

ANADIR = Path(os.environ.get("LECTURE_DATA", "/opt/lecture-data")) / "txs0506"

os.environ['FERMI_DIFFUSE_DIR'] = os.path.join(
    os.environ['CONDA_PREFIX'],
    'share', 'fermitools', 'refdata', 'fermi', 'galdiffuse')

os.chdir(ANADIR)
print("working in:", os.getcwd())
print("\ncontents:")
for p in sorted(ANADIR.iterdir()):
    print(f"  {p.name}")


The pre-written `ft1.txt`, `ft2.txt`, `config.yaml` are the only things this notebook *needs*; everything else (`*_00.fits`, `srcmap_00.fits`, `ltcube_00.fits`, …) will be created by `gta.setup()` and re-used on subsequent runs.


In [ ]:
print("=== ft1.txt ===")
print(open("ft1.txt").read())
print("=== ft2.txt ===")
print(open("ft2.txt").read())
print("=== config.yaml ===")
print(open("config.yaml").read())


## Run the analysis chain

`gta.setup()` will run `gtselect → gtmktime → gtltcube → gtbin → gtexpcube2 → gtsrcmaps`. With 2 years of data over a 15° ROI, expect **~15-30 min** the *first* time (dominated by `gtltcube` and `gtsrcmaps`). Re-runs find the cached files and finish in seconds.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from fermipy.gtanalysis import GTAnalysis
gta = GTAnalysis("config.yaml", logging={"verbosity": 3})
gta.setup()


### ROI inventory

`print_roi()` after `setup()` lists every source fermipy has loaded into the model. Note the difference from PG 1553's ROI: **more catalog sources at small offset**, and the diffuse `npred` (counts predicted from `gll_iem_v07`) is larger because of the lower latitude.


In [ ]:
gta.print_roi()
print()
print(gta.roi['4FGL J0509.4+0542'])


### Visual

Let's look ar our counts map...

In [ ]:
from astropy.io import fits
from astropy.wcs import WCS

h = fits.open('ccube.fits')
wcs = WCS(h[0].header).dropaxis(-1)
counts = h[0].data

fig, ax = plt.subplots(subplot_kw={'projection': wcs}, figsize=(7, 6))
im = ax.imshow(np.sum(counts, axis=0), origin='lower',
               cmap='plasma', interpolation='nearest')
plt.colorbar(im, ax=ax, label='counts')
ax.set_xlabel('R.A. (deg)'); ax.set_ylabel('Dec. (deg)')
ax.grid()
plt.show()


## Likelihood fit

Same as PG 1553:

1. Fix everything,
2. Free *normalizations* of sources within 3° of the centre,
3. Free the diffuse normalizations,
4. Free the **full spectral shape** of TXS (LogParabola: norm, α, β),
5. Fit.


In [ ]:
gta.free_sources(free=False)
gta.free_sources(distance=3.0, pars='norm')
gta.free_source('galdiff', pars='norm')
gta.free_source('isodiff')
gta.free_source('4FGL J0509.4+0542')

fit = gta.fit()
print('fit_quality:', fit['fit_quality'])
print(gta.roi['4FGL J0509.4+0542'])

gta.write_roi('fit_txs', make_plots=True)


## SED — average over the 2-year window


In [ ]:
gta.load_roi('fit_txs')
sed = gta.sed('4FGL J0509.4+0542')


In [ ]:
c       = np.load('fit_txs.npy', allow_pickle=True).flat[0]
mflux   = c['sources']['4FGL J0509.4+0542']['model_flux']
E       = np.array(mflux['energies'])
dnde    = np.array(mflux['dnde'])
dnde_hi = np.array(mflux['dnde_hi'])
dnde_lo = np.array(mflux['dnde_lo'])

fig, ax = plt.subplots(figsize=(7, 5))
ax.set_xscale('log'); ax.set_yscale('log')
ax.plot(E, E**2 * dnde, 'k-', lw=1.5, label='best fit (LogParabola)')
ax.fill_between(E, E**2 * dnde_lo, E**2 * dnde_hi,
                color='k', alpha=0.2, label=r'$\pm 1\sigma$')

# SED points: detections (TS >= 4) and 95% ULs otherwise
e_ctr  = np.array(sed['e_ctr'])
e2dnde = np.array(sed['e2dnde'])
e2err  = np.array(sed['e2dnde_err'])
e2ul   = np.array(sed['e2dnde_ul95'])
det    = np.array(sed['ts']) >= 4

ax.errorbar(e_ctr[det], e2dnde[det], yerr=e2err[det],
            fmt='o', color='C0', label='SED points (TS≥4)')
ax.errorbar(e_ctr[~det], e2ul[~det], yerr=0.3*e2ul[~det],
            uplims=True, fmt='v', color='C0', alpha=0.6, label='95% UL')

ax.set_xlabel('E [MeV]')
ax.set_ylabel(r'$E^2\,dN/dE$ [MeV cm$^{-2}$ s$^{-1}$]')
ax.set_title('TXS 0506+056 — 2016-09 → 2018-09 average')
ax.legend()
fig.tight_layout()
plt.show()


## Lightcurve around IC-170922A

This is the result we are interested in! We bin the 2-year window into **monthly slices** (~30 d) and re-fit TXS's normalization in each bin (everything else fixed at the global best-fit).

24 bins × one fit each — and **sequential**, not parallel: on macOS, `multithread=True` triggers a circular-import crash in fermipy's spawn-based pool (`fermipy.lightcurve` ↔ `fermipy.gtanalysis`). Sequential takes ~30-45 min the first time; subsequent runs reuse the per-bin source maps and are much faster.

`gta.lightcurve` does the bookkeeping for you: per-bin `gtbin`/`gtsrcmaps` if needed, per-bin fit, per-bin TS, flux, error, upper limit.


In [ ]:
binsize_days = 30.0
tneutrino    = 527806473.0   # MET for 2017-09-22T20:54:30 UTC (IC-170922A)
tmin, tmax   = 494380804, 557452805

edges     = np.arange(tmin, tmax + 1, binsize_days * 86400.)
time_bins = np.column_stack([edges[:-1], edges[1:]])
print(f"{len(time_bins)} bins of ~{binsize_days:.0f} days")

# multithread=False: macOS Python 3.9+ uses 'spawn' for multiprocessing,
# which trips a circular import in fermipy.lightcurve <-> fermipy.gtanalysis.
# Sequential is slower (~30-45 min for 24 bins) but works.
lc = gta.lightcurve('4FGL J0509.4+0542',
                    free_params=['Prefactor'],
                    time_bins=edges.tolist(),
                    multithread=False,
                    save_bin_data=False)


In [ ]:
from astropy.time import Time

tmean = 0.5 * (np.asarray(lc['tmin']) + np.asarray(lc['tmax']))
flux  = np.asarray(lc['flux'])
ferr  = np.asarray(lc['flux_err'])
ts    = np.asarray(lc['ts'])
ful   = np.asarray(lc['flux_ul95'])
det   = ts >= 4

# MET → ISO for a readable x-axis (Fermi MET epoch: 2001-01-01 00:00:00 TT)
def met_to_mjd(met):
    return 51910.0 + (met + 64.184) / 86400.0   # rough; ok for plotting
mjd  = met_to_mjd(tmean)
mjd_n = met_to_mjd(tneutrino)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.errorbar(mjd[det], flux[det], yerr=ferr[det],
            xerr=binsize_days/2, fmt='o', color='C0',
            label='monthly flux (TS≥4)')
ax.errorbar(mjd[~det], ful[~det], yerr=0.3*ful[~det],
            xerr=binsize_days/2, uplims=True, fmt='v', color='C0',
            alpha=0.5, label='95% UL')

ax.axvline(mjd_n, color='C3', ls='--', lw=1.5, label='IC-170922A')
ax.set_yscale('log')
ax.set_xlabel('MJD')
ax.set_ylabel(r'F(>100 MeV) [cm$^{-2}$ s$^{-1}$]')
ax.set_title('TXS 0506+056 — Fermi-LAT monthly lightcurve')
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()


The bin containing **2017-09-22** (and the few months around it) sits well above the rest — the ~6-month enhancement reported in the *Science* 361 paper. 

### What we could do next

- **Finer binning** over the flare window (weekly or daily) to constrain the variability time-scale and feed it into a size argument for the emitting region (R ≲ c · t_var · δ).
- **Bayesian-block** segmentation to avoid the arbitrary fixed binning.
- **Time-resolved SED** in the flare bin only — that's the spectrum to feed into a multi-wavelength SED fit. 
- **EBL absorption** model on the intrinsic spectrum if you care about >100 GeV.

### Why this was non-trivial — recap

- The lower Galactic latitude made the diffuse background brighter and asymmetric across the ROI
- The source field is busier than at high \|b\|
- The source is variable: the 2-year-averaged SED averages a quiescent state with a 6-month flare. The lightcurve makes the result interpretable.